# Question 3: Model Performance Analysis

This notebook evaluates the models implemented in Questions 1 and 2.

## Metrics used
- **Regression:** MAE, MSE, RMSE, and R?; lower error and higher R? are better.
- **Classification:** accuracy, precision, recall, F1-score, and ROC-AUC.
- **Clustering:** silhouette score and Adjusted Rand Index.
- **PCA:** explained variance and reconstruction MSE.

The notebook uses built-in scikit-learn datasets and saves result tables and a comparison chart in `question_3_results`.

In [2]:
"""Question 3: performance analysis of the models from Questions 1 and 2.

The script evaluates:
* regression: Linear, Ridge, and Lasso regression;
* classification: decision tree, logistic regression (L2 and L1),
  KNN, Gaussian Naive Bayes, bagging, random forest, and AdaBoost;
* clustering: K-Means, hierarchical clustering, and Gaussian mixtures;
* dimensionality reduction: PCA.

The datasets are built into scikit-learn, so the file can be run without
downloading external data:

    python 03_model_performance_analysis.py
"""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.datasets import load_breast_cancer, load_diabetes, load_iris
from sklearn.decomposition import PCA
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
    silhouette_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier


RANDOM_STATE = 42
OUTPUT_DIR = Path.cwd() / "question_3_results"


def evaluate_regression() -> pd.DataFrame:
    """Compare regression models using error and goodness-of-fit metrics."""
    data = load_diabetes(as_frame=True)
    X_train, X_test, y_train, y_test = train_test_split(
        data.data,
        data.target,
        test_size=0.2,
        random_state=RANDOM_STATE,
    )

    models = {
        "Linear Regression": LinearRegression(),
        "Ridge Regression": Ridge(alpha=1.0),
        "Lasso Regression": Lasso(alpha=0.01, max_iter=20_000),
    }
    rows = []
    for name, estimator in models.items():
        model = Pipeline(
            [("scale", StandardScaler()), ("model", estimator)]
        )
        model.fit(X_train, y_train)
        prediction = model.predict(X_test)
        rows.append(
            {
                "Model": name,
                "MAE": mean_absolute_error(y_test, prediction),
                "MSE": mean_squared_error(y_test, prediction),
                "RMSE": np.sqrt(mean_squared_error(y_test, prediction)),
                "R2": r2_score(y_test, prediction),
            }
        )

    return pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)


def evaluate_classification() -> pd.DataFrame:
    """Compare classifiers using accuracy, F1, and ROC-AUC."""
    data = load_breast_cancer(as_frame=True)
    X_train, X_test, y_train, y_test = train_test_split(
        data.data,
        data.target,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=data.target,
    )

    models = {
        "Decision Tree (Gini)": DecisionTreeClassifier(
            criterion="gini", max_depth=5, random_state=RANDOM_STATE
        ),
        "Decision Tree (Entropy)": DecisionTreeClassifier(
            criterion="entropy", max_depth=5, random_state=RANDOM_STATE
        ),
        "Logistic Regression (L2/Ridge)": Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        penalty="l2", solver="lbfgs", max_iter=5_000, random_state=RANDOM_STATE
                    ),
                ),
            ]
        ),
        "Logistic Regression (L1/Lasso)": Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        penalty="l1", solver="liblinear", max_iter=5_000, random_state=RANDOM_STATE
                    ),
                ),
            ]
        ),
        "K-Nearest Neighbours": Pipeline(
            [
                ("scale", StandardScaler()),
                ("model", KNeighborsClassifier(n_neighbors=5)),
            ]
        ),
        "Gaussian Naive Bayes": GaussianNB(),
        "Bagging": BaggingClassifier(
            n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
        ),
        "AdaBoost": AdaBoostClassifier(
            n_estimators=100, random_state=RANDOM_STATE
        ),
    }

    rows = []
    for name, model in models.items():
        model.fit(X_train, y_train)
        prediction = model.predict(X_test)
        probability = model.predict_proba(X_test)[:, 1]
        rows.append(
            {
                "Model": name,
                "Accuracy": accuracy_score(y_test, prediction),
                "Precision": precision_score(y_test, prediction),
                "Recall": recall_score(y_test, prediction),
                "F1": f1_score(y_test, prediction),
                "ROC-AUC": roc_auc_score(y_test, probability),
            }
        )

    return pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)


def evaluate_clustering() -> pd.DataFrame:
    """Compare clustering quality with internal and external metrics."""
    data = load_iris(as_frame=True)
    X = StandardScaler().fit_transform(data.data)
    y = data.target

    models = {
        "K-Means": KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE),
        "Hierarchical (Ward)": AgglomerativeClustering(n_clusters=3, linkage="ward"),
        "Hierarchical (Complete)": AgglomerativeClustering(
            n_clusters=3, linkage="complete"
        ),
        "Gaussian Mixture": GaussianMixture(
            n_components=3, n_init=10, random_state=RANDOM_STATE
        ),
    }

    rows = []
    for name, model in models.items():
        labels = model.fit_predict(X)
        rows.append(
            {
                "Model": name,
                "Silhouette": silhouette_score(X, labels),
                "Adjusted Rand Index": adjusted_rand_score(y, labels),
                "Number of clusters": len(np.unique(labels)),
            }
        )

    return pd.DataFrame(rows).sort_values(
        "Silhouette", ascending=False
    ).reset_index(drop=True)


def evaluate_pca() -> pd.DataFrame:
    """Measure PCA compression using explained variance and reconstruction error."""
    data = load_breast_cancer(as_frame=True)
    X_scaled = StandardScaler().fit_transform(data.data)
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(X_scaled)
    reconstructed = pca.inverse_transform(reduced)

    return pd.DataFrame(
        [
            {
                "Method": "PCA (2 components)",
                "Original features": X_scaled.shape[1],
                "Reduced features": reduced.shape[1],
                "Explained variance": pca.explained_variance_ratio_.sum(),
                "Reconstruction MSE": mean_squared_error(X_scaled, reconstructed),
            }
        ]
    )


def save_visualizations(
    regression: pd.DataFrame,
    classification: pd.DataFrame,
    clustering: pd.DataFrame,
) -> None:
    """Save compact comparison charts for the report."""
    OUTPUT_DIR.mkdir(exist_ok=True)
    figure, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].bar(regression["Model"], regression["RMSE"], color="steelblue")
    axes[0].set_title("Regression: RMSE (lower is better)")
    axes[0].tick_params(axis="x", rotation=70)

    axes[1].bar(classification["Model"], classification["F1"], color="darkorange")
    axes[1].set_title("Classification: F1 (higher is better)")
    axes[1].tick_params(axis="x", rotation=70)

    axes[2].bar(clustering["Model"], clustering["Silhouette"], color="seagreen")
    axes[2].set_title("Clustering: silhouette (higher is better)")
    axes[2].tick_params(axis="x", rotation=70)

    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "model_comparison.png", dpi=150)
    plt.close(figure)


def main() -> None:
    regression = evaluate_regression()
    classification = evaluate_classification()
    clustering = evaluate_clustering()
    pca = evaluate_pca()

    OUTPUT_DIR.mkdir(exist_ok=True)
    regression.to_csv(OUTPUT_DIR / "regression_metrics.csv", index=False)
    classification.to_csv(OUTPUT_DIR / "classification_metrics.csv", index=False)
    clustering.to_csv(OUTPUT_DIR / "clustering_metrics.csv", index=False)
    pca.to_csv(OUTPUT_DIR / "pca_metrics.csv", index=False)
    save_visualizations(regression, classification, clustering)

    print("\nREGRESSION PERFORMANCE")
    print(regression.round(4).to_string(index=False))
    print("\nCLASSIFICATION PERFORMANCE")
    print(classification.round(4).to_string(index=False))
    print("\nCLUSTERING PERFORMANCE")
    print(clustering.round(4).to_string(index=False))
    print("\nPCA PERFORMANCE")
    print(pca.round(4).to_string(index=False))
    print(f"\nResults saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()



REGRESSION PERFORMANCE
            Model     MAE       MSE    RMSE     R2
 Ridge Regression 42.8120 2892.0146 53.7775 0.4541
 Lasso Regression 42.7950 2898.3680 53.8365 0.4529
Linear Regression 42.7941 2900.1936 53.8534 0.4526

CLASSIFICATION PERFORMANCE
                         Model  Accuracy  Precision  Recall     F1  ROC-AUC
Logistic Regression (L1/Lasso)    0.9912     0.9863  1.0000 0.9931   0.9967
Logistic Regression (L2/Ridge)    0.9825     0.9861  0.9861 0.9861   0.9954
                      AdaBoost    0.9561     0.9467  0.9861 0.9660   0.9818
                 Random Forest    0.9561     0.9589  0.9722 0.9655   0.9931
          K-Nearest Neighbours    0.9561     0.9589  0.9722 0.9655   0.9788
          Gaussian Naive Bayes    0.9386     0.9452  0.9583 0.9517   0.9878
                       Bagging    0.9386     0.9452  0.9583 0.9517   0.9927
       Decision Tree (Entropy)    0.9298     0.9706  0.9167 0.9429   0.9435
          Decision Tree (Gini)    0.9211     0.9565  0.9167 